In [ ]:
#|default_exp cli

# CLI
> Command-line interface for running seootter commands and reports.

In [ ]:
#| export
from fastcore.script import call_parse

from seootter.gsc_client import GSCAuth, get_date_range
from seootter.gsc.sync import get_missing_dates, store_date_range, sync_missing_dates
from seootter.gsc.queries import get_top_pages, get_wins, compare_date_ranges, get_country_breakdown, get_top_queries
from seootter.index_tracking import fetch_sitemap_urls, store_index_status, get_not_indexed_by_reason, get_index_status, \
    get_not_indexed_pages, get_index_history, store_all_index_status

from seootter.keyword_ranking import get_keyword_rankings, get_keyword_rankings_daily
from seootter.schema_extractor import extract_faq_queries
from seootter.schema_validator import validate_page
from seootter.report.articles import sync_articles_to_db
from seootter.report.analysis import analyze_article
from seootter.report.generator import generate_seo_report, print_issues_report
from seootter.seo_site_analysis import analyze_content_groups, analyze_keyword_cannibalization, \
    analyze_content_groups_fast
from seootter.sqlite_db import get_session
from pathlib import Path
from seootter.article import Article

CONFIG_DIR = Path.home() / ".config" / "seootter"

In [ ]:
#|export
@call_parse
def seo_otter_sync(
        site_url: str,  # e.g. "sc-domain:example.com"
        days: int = 30,  # how many days back to sync
        secrets: str = str(Path.home() / ".config" / "seootter" / "client_secrets.json"),

        token: str = "./token.pickle"
):
    "Sync GSC data for a site"
    auth = GSCAuth(secrets_file=secrets, token_file=token)
    try:
        auth.get_credentials()
    except ValueError:
        print("No credentials found, starting authentication...")
        auth.authenticate()
    with get_session() as session:
        start_date, end_date = get_date_range("last_days", days=days)
        results = sync_missing_dates(session, auth, site_url, start_date, end_date)
    print(f"✅ Synced {len(results['successful'])} days, {results['total_records']} records")




In [ ]:
#| export
from seootter.models import Website, add_or_update_website, get_all_websites, print_websites, delete_website, \
    GSCAnalytics

import importlib.util


def load_mapper(domain: str) -> dict[str, str]:
    "Load and run the mapper for a given domain."
    mapper_path = Path.home() / ".config" / "seootter" / "mappers" / domain / "mapper.py"
    if not mapper_path.exists():
        raise FileNotFoundError(
            f"❌ No mapper found for {domain}\n"
            f"👉 Create one at: {mapper_path}"
        )
    spec = importlib.util.spec_from_file_location("mapper", mapper_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.get_url_file_mapping()


@call_parse
def seo_otter_report(
        website_id: int,  # Website ID from the database
        days: int = 90,  # Days of GSC data to include
        insights: bool = False  # Include query trends and green keywords
):
    "Generate SEO report for a website"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return

        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")

        try:
            url_file_mapping = load_mapper(domain)
        except FileNotFoundError as e:
            print(e)
            return

        sync_articles_to_db(session, website_id, url_file_mapping)

        is_quarto = website.site_type == "quarto"
        report = generate_seo_report(
            session=session,
            website_id=website_id,
            domain=domain,
            is_quarto=is_quarto,
            include_insights=insights,
            days=days,
            title_is_h1=is_quarto
        )
    print_issues_report(report)



In [ ]:
#| export
@call_parse
def seo_otter_audit(
        website_id: int,  # Website ID from the database
        url: str = "",  # Page URL to audit
        file_path: str = "",  # Local file path to audit
        days: int = 90,  # Days of GSC data to include
        insights: bool = False  # Include trends and green keywords
):
    "Audit a single page for SEO issues"
    if not url and not file_path:
        print("Provider either --url or --file-path")
        return
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"Website {website_id}  not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        if not file_path:
            try:
                mapping = load_mapper(domain)
            except FileNotFoundError as e:
                print(e)
                return
            file_path = mapping.get(url)
            if not file_path:
                print(f"No file found for URL: {url}")
                return
    is_quarto = website.site_type == "quarto"
    article = Article(website_id=website_id, file_path=file_path, url=url)
    report = analyze_article(article, domain, is_quarto, title_is_h1=is_quarto)
    return report



In [ ]:
#| export
from rich.table import Table
from rich import print as rprint


def print_rankings(results: list[dict]) -> None:
    "Print keyword rankings as a rich table."
    table = Table(title="Keyword Rankings")
    table.add_column("Keyword", style="cyan")
    table.add_column("Position", justify="right")
    table.add_column("Change", justify="right")
    table.add_column("Clicks", justify="right")
    table.add_column("Impressions", justify="right")
    for r in results:
        change = r["change"]
        change_str = f"[green]{change:+.1f}[/green]" if change and change < 0 else f"[red]{change:+.1f}[/red]" if change else "-"
        table.add_row(r["keyword"], str(r["position"] or "-"), change_str, str(r["clicks"]), str(r["impressions"]))
    rprint(table)


@call_parse
def seo_otter_rank(
        website_id: int,  # Website ID from the database
        keywords: str,  # Comma-separated keywords to tracke
        range_type: str = "entire_history",  # Date range type
        country: str = ""  # Optional country code (e.g. 'sau')
):
    "Check keyword rankings for a website"
    keyword_list = [k.strip() for k in keywords.split(",")]
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return

        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        country = country or None
        keywords_ranking = get_keyword_rankings(session, site_url, keyword_list, range_type=range_type, country=country)
    print_rankings(keywords_ranking)




In [ ]:
#| export
def print_top_pages(rows: list[dict], country: str | None = None) -> None:
    "Print top pages as a rich table."
    table = Table(title="Top Pages")
    table.add_column("Page", style="cyan")
    table.add_column("Clicks", justify="right")
    table.add_column("Impressions", justify="right")
    table.add_column("Avg Position", justify="right")
    table.add_column("CTR", justify="right")
    if country:
        table.add_column("Country", justify="right")
    for r in rows:
        row_data = [
            r["page"],
            str(r["total_clicks"]),
            str(r["total_impressions"]),
            f"{r['avg_position']:.1f}" if r["avg_position"] else "-",
            f"{r['avg_ctr'] * 100:.1f}%" if r["avg_ctr"] else "-"
        ]
        if country:
            row_data.append(country)
        table.add_row(*row_data)
    rprint(table)


@call_parse
def seo_otter_top_pages(
        website_id: int,  # Website ID from the database
        range_type: str = "entire_history",  # Date range type
        limit: int = 20,  # Number of pages to show
        country: str = ""  # Optional country code (e.g. 'sau')
):
    "Show top performing pages by clicks and impressions"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        start, end = get_date_range(range_type=range_type)
        rows = get_top_pages(session, site_url, start, end,
                             country=country or None, limit=limit, sort_by="clicks")
    print_top_pages(rows, country or None)



In [ ]:
#| export
def print_wins(rows: list[dict]) -> None:
    "Print keyword wins as a rich table."
    table = Table(title="🏆 Keyword Wins")
    table.add_column("Keyword", style="cyan")
    table.add_column("Impressions", justify="right")
    table.add_column("Clicks", justify="right")
    table.add_column("Avg Position", justify="right")
    table.add_column("CTR", justify="right")
    for r in rows:
        table.add_row(
            r["query"],
            str(r["total_impressions"]),
            str(r["total_clicks"]),
            f"{r['avg_position']:.1f}" if r["avg_position"] else "-",
            f"{r['avg_ctr'] * 100:.1f}%" if r["avg_ctr"] else "-"
        )
    rprint(table)


@call_parse
def seo_otter_wins(
        website_id: int,  # Website ID from the database
        page_url: str = "",  # Optional page URL to filter
        range_type: str = "entire_history",  # Date range type
        min_impressions: int = 100,  # Minimum impressions threshold
        min_position: float = 10.0,  # Minimum avg position
        max_position: float = 50.0,  # Maximum avg position cutoff
        limit: int = 20,  # Number of results to show
        country: str = ""  # Optional country code
):
    "Show high-impression, low-ranking keyword wins"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        start, end = get_date_range(range_type=range_type)
        rows = get_wins(session, site_url, start, end,
                        min_impressions=min_impressions,
                        min_position=min_position,
                        max_position=max_position,
                        country=country or None,
                        page_url=page_url or None,
                        limit=limit)
    print_wins(rows)

In [ ]:
#| export
def print_canob_keyword(result: dict) -> None:
    "Print keyword cannibalization result as a rich table."
    if not result["has_cannibalization"]:
        rprint(f"[green]✅ No cannibalization found for '{result['keyword']}'[/green]")
        return
    table = Table(title=f"⚠️ Cannibalization: '{result['keyword']}' ({result['count']} pages)")
    table.add_column("File Path", style="cyan")
    table.add_column("Keyword Count", justify="right")
    table.add_column("Density", justify="right")
    for a in result["articles"]:
        table.add_row(a["file_path"], str(a["count"]), f"{a['density']:.2f}%")
    rprint(table)


def print_canob_groups(result: dict) -> None:
    "Print content group duplicates as a rich table."
    if not result["duplicate_groups"]:
        rprint("[green]✅ No duplicate content groups found[/green]")
        return
    table = Table(title=f"⚠️ Duplicate Content Groups ({result['duplicate_groups']} groups)")
    table.add_column("Main Article", style="cyan")
    table.add_column("Similar Article", style="yellow")
    table.add_column("Similarity", justify="right")
    for group in result["groups"]:
        for similar in group["similar_articles"]:
            table.add_row(group["main_article"], similar["file_path"], f"{similar['similarity']:.0%}")
    rprint(table)


@call_parse
def seo_otter_canob(
        website_id: int,  # Website ID from the database
        keyword: str = "",  # Specific keyword to check (omit for full site scan)
        similarity_threshold: float = 0.8  # Similarity threshold for content groups
):
    "Detect keyword cannibalization across pages"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return

        if keyword:
            result = analyze_keyword_cannibalization(session, website_id, keyword)
            print_canob_keyword(result)
        else:
            result = analyze_content_groups_fast(session, website_id, similarity_threshold)
            print_canob_groups(result)


## Indexing

In [ ]:
#| export
from seootter.index_tracking import IndexStatus


def fmt_date(dt: str | None) -> str:
    "Format ISO datetime to YYYY-MM-DD."
    return dt[:10] if dt else "-"


def print_index_check(status: IndexStatus) -> None:
    color = "green" if status.verdict == "PASS" else "red"
    rprint(f"[{color}]● {status.verdict}[/{color}] — {status.page_url}")
    rprint(f"  Coverage : {status.coverage_state}")
    rprint(f"  Indexing : {status.indexing_state}")
    rprint(f"  Robots   : {status.robots_txt_state}")
    rprint(f"  Last crawl: {fmt_date(status.last_crawl_time)}")


def print_index_report(all_pages: list, grouped: dict) -> None:
    passed = sum(1 for p in all_pages if p.verdict == "PASS")
    rprint(
        f"\n📊 Total: {len(all_pages)} pages — [green]{passed} indexed[/green], [red]{len(all_pages) - passed} not indexed[/red]\n")
    for reason, pages in grouped.items():
        table = Table(title=f"⚠️ {reason} ({len(pages)})")
        table.add_column("Page URL", style="cyan")
        table.add_column("Last Crawl")
        for p in pages:
            table.add_row(p.page_url, fmt_date(p.last_crawl_time) or "-")
        rprint(table)


def print_crawl_errors(pages: list) -> None:
    if not pages:
        rprint("[green]✅ No crawl errors found[/green]")
        return
    table = Table(title=f"🚨 Crawl Issues ({len(pages)} pages)")
    table.add_column("Page URL", style="cyan")
    table.add_column("Verdict", justify="right")
    table.add_column("Coverage State", justify="right")
    table.add_column("Last Crawl", justify="right")
    for p in pages:
        table.add_row(p.page_url, p.verdict, p.coverage_state or "-", fmt_date(p.last_crawl_time) or "-")
    rprint(table)


In [ ]:


#| export
from seootter.index_tracking import store_all_index_status


@call_parse
def seo_otter_index_check(
        website_id: int,  # Website ID from the database
        page_url: str  # Page URL to inspect
):
    "Check and store index status for a single page"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        auth = GSCAuth()
        store_index_status(session, auth, site_url, page_url)
        history = get_index_history(session, page_url)
    print_index_check(history[0])


@call_parse
def seo_otter_index_report(
        website_id: int,  # Website ID from the database
        sitemap_url: str = ""  # Sitemap URL to refresh data (optional)
):
    "Show index status report grouped by reason"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        if sitemap_url:
            auth = GSCAuth()
            store_all_index_status(session, auth, site_url, sitemap_url)
        grouped = get_not_indexed_by_reason(session, site_url)
        all_pages = get_index_status(session, site_url)
    print_index_report(all_pages, grouped)


@call_parse
def seo_otter_crawl_errors(
        website_id: int  # Website ID from the database
):
    "Show pages with crawl or indexing issues"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        pages = get_not_indexed_pages(session, site_url)
    print_crawl_errors(pages)


@call_parse
def seo_otter_index_refresh(
        website_id: int,  # Website ID from the database
        sitemap_url: str  # Sitemap URL to fetch and re-check
):
    "Refresh index status for all pages in a sitemap"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        auth = GSCAuth()
        results = store_all_index_status(session, auth, site_url, sitemap_url)
    rprint(f"[green]✅ {len(results['successful'])} pages refreshed[/green]")
    if results["failed"]:
        rprint(f"[red]❌ {len(results['failed'])} failed[/red]")


## Trends & Comparsion

In [ ]:
#| export
from datetime import datetime


def print_trend(keyword: str, rows: list[dict]) -> None:
    if not rows:
        rprint(f"[red]No data found for '{keyword}'[/red]")
        return
    table = Table(title=f"📈 Trend: {keyword}")
    table.add_column("Date", style="cyan")
    table.add_column("Position", justify="right")
    table.add_column("Clicks", justify="right")
    table.add_column("Impressions", justify="right")
    for r in rows:
        table.add_row(str(r["date"]), f"{r['position']:.1f}", str(r["clicks"]), str(r["impressions"]))
    rprint(table)


def aggregate_by_period(rows: list[dict], range_type: str, months: int = 3) -> list[dict]:
    "Aggregate daily rows by week or month depending on range."
    from collections import defaultdict

    if range_type in ("last_7_days", "last_days", "today"):
        return rows  # daily

    use_monthly = range_type == "entire_history" or (range_type == "last_months" and months > 3)

    groups = defaultdict(list)
    for r in rows:
        date = r["date"] if isinstance(r["date"], str) else str(r["date"])
        if use_monthly:
            key = date[:7]  # YYYY-MM
        else:
            key = f"{date[:4]}-W{datetime.strptime(date, '%Y-%m-%d').strftime('%W')}"

        groups[key].append(r)

    return [{"date": period,
             "position": sum(i["position"] for i in items) / len(items),
             "clicks": sum(i["clicks"] for i in items),
             "impressions": sum(i["impressions"] for i in items)}
            for period, items in sorted(groups.items())]


@call_parse
def seo_otter_trend(
        website_id: int,  # Website ID from the database
        keyword: str,  # Keyword to track
        range_type: str = "entire_history",  # Date range type
        months: int = 3,  # Months for last_months range type
        days: int = 7,  # Days for last_days range type
        country: str = ""  # Optional country code
):
    "Show position trend for a keyword aggregated by period"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            print(f"❌ Website {website_id} not found")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        rows = get_keyword_rankings_daily(session, site_url, [keyword],
                                          range_type=range_type,
                                          months=months, days=days,
                                          country=country or None)
        rows = aggregate_by_period(rows, range_type, months=months)
    print_trend(keyword, rows)



In [ ]:
#| export
@call_parse
def seo_otter_add_website(
        url: str,  # Website URL (e.g., https://example.com)
        name: str,  # Display name for the website
        site_type: str,  # Type: quarto, astro, wordpress, etc.
        desc: str = "",  # Optional description
        lang: str = "en",  # Language code (e.g., en, ar)
        content_dir: str = "",  # Local path to content directory
        website_id: int = None  # ID to update existing website (omit to create new)
):
    "Add a new website or update existing by ID"
    try:
        with get_session() as session:
            website = add_or_update_website(
                session, url=url, name=name, site_type=site_type,
                desc=desc, lang=lang, content_dir=content_dir,
                website_id=website_id
            )
        action = "updated" if website_id else "added"
        rprint(f"[green]✅ Website {action}:[/green] {website.name} (ID: {website.id})")
    except ValueError as e:
        rprint(f"[red]❌ Error: {e}[/red]")

In [ ]:
#| export
@call_parse
def seo_otter_list_websites():
    "List all registered websites"
    with get_session() as session:
        websites = get_all_websites(session)
    print_websites(websites)

In [ ]:
#| export
@call_parse
def seo_otter_delete_website(
        website_id: int  # Website ID to delete
):
    "Delete a website by ID (irreversible!)"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            rprint(f"[red]❌ Website {website_id} not found[/red]")
            return

        try:
            delete_website(session, website_id)
            rprint(f"[green]✅ Deleted:[/green] {website.name} (ID: {website_id})")
        except Exception as e:
            rprint(f"[red]❌ Error: {e}[/red]")

In [ ]:
#| eval: false
# Test: List all websites
seo_otter_list_websites()

# Test: Add a new website
seo_otter_add_website(
    url="https://test-example.com",
    name="Test Site",
    site_type="astro",
    desc="A test website",
    lang="en"
)
seo_otter_list_websites()
seo_otter_delete_website(5)
seo_otter_list_websites()

In [ ]:
#| export
def print_comparison(result: dict) -> None:
    "Print date range comparison as a rich table."
    p1 = result["period1"]
    p2 = result["period2"]

    title = "📊 Period Comparison"
    if result.get("page_url"):
        title = f"📊 Page Comparison: {result['page_url']}"

    table = Table(title=title)
    table.add_column("Metric")
    table.add_column(f"{p1['start']} to {p1['end']}", justify="right")
    table.add_column(f"{p2['start']} to {p2['end']}", justify="right")
    table.add_column("Change", justify="right")

    clicks_change = p2["clicks"] - p1["clicks"]
    impr_change = p2["impressions"] - p1["impressions"]
    pos_change = p2["avg_position"] - p1["avg_position"]

    def fmt_change(val, reverse=False):
        color = "green" if (val > 0 and not reverse) or (val < 0 and reverse) else "red"
        return f"[{color}]{val:+,}[/{color}]"

    table.add_row("Clicks", str(p1["clicks"]), str(p2["clicks"]), fmt_change(clicks_change))
    table.add_row("Impressions", str(p1["impressions"]), str(p2["impressions"]), fmt_change(impr_change))
    table.add_row("Avg Position", str(p1["avg_position"]), str(p2["avg_position"]),
                  fmt_change(pos_change, reverse=True))
    table.add_row("Avg CTR %", str(p1["avg_ctr"]), str(p2["avg_ctr"]), "-")

    rprint(table)


@call_parse
def seo_otter_compare(
        website_id: int,  # Website ID from the database
        start1: str,  # First period start (YYYY-MM-DD)
        end1: str,  # First period end (YYYY-MM-DD)
        start2: str,  # Second period start (YYYY-MM-DD)
        end2: str,  # Second period end (YYYY-MM-DD)
        page_url: str = ""  # Optional specific page URL to compare
):
    "Compare GSC metrics between two date ranges, optionally for a specific page"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            rprint(f"[red]❌ Website {website_id} not found[/red]")
            return

        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"

        result = compare_date_ranges(session, site_url, start1, end1, start2, end2,
                                     page_url=page_url or None)
    print_comparison(result)

In [ ]:
#| eval: false
# Test: Compare two date ranges (use a real website_id from your DB)
seo_otter_compare(
    website_id=4,  # Change to real ID
    start1="2025-03-01",
    end1="2025-03-31",
    start2="2025-04-01",
    end2="2025-04-05",
    page_url="https://kareemai.com/"  # Change to real URL or omit for site-wide
)


In [ ]:
#| export
def print_country_breakdown(rows: list[dict]) -> None:
    "Print country breakdown as a rich table."
    if not rows:
        rprint("[yellow]No country data found[/yellow]")
        return

    table = Table(title="🌍 Traffic by Country")
    table.add_column("Country", style="cyan")
    table.add_column("Clicks", justify="right")
    table.add_column("Impressions", justify="right")
    table.add_column("Avg Position", justify="right")
    table.add_column("CTR", justify="right")

    total_clicks = sum(r["clicks"] for r in rows)

    for r in rows:
        pct = (r["clicks"] / total_clicks * 100) if total_clicks > 0 else 0
        table.add_row(
            r["country"] or "Unknown",
            f"{r['clicks']:,} ({pct:.1f}%)",
            f"{r['impressions']:,}",
            f"{r['avg_position']:.1f}" if r["avg_position"] else "-",
            f"{(r['avg_ctr'] or 0) * 100:.1f}%"
        )

    rprint(table)
    rprint(f"\n[dim]Total: {total_clicks:,} clicks across {len(rows)} countries[/dim]")


@call_parse
def seo_otter_country_breakdown(
    website_id: int,  # Website ID from the database
    range_type: str = "last_month",  # Date range type
    start_date: str = "",  # Custom start date (YYYY-MM-DD)
    end_date: str = "",  # Custom end date (YYYY-MM-DD)
    page_url: str = "",  # Optional specific page URL to filter
    limit: int = 20  # Number of top countries to show
):
    "Show traffic breakdown by country, optionally for a specific page"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            rprint(f"[red]❌ Website {website_id} not found[/red]")
            return

        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"

        # Determine date range
        if start_date and end_date:
            start, end = start_date, end_date
        else:
            start, end = get_date_range(range_type=range_type)

        rows = get_country_breakdown(session, site_url, start, end,
                                      page_url=page_url or None,
                                      limit=limit)
    print_country_breakdown(rows)

In [ ]:
#| eval: false
# Test: Country breakdown (use a real website_id from your DB)
seo_otter_country_breakdown(
    website_id=1,  # Change to real ID
    limit=20
)

seo_otter_country_breakdown(
    website_id=4,  # Change to real ID
    range_type="last_month",
    page_url="https://kareemai.com/blog/posts/speech_recognition/muaalm_quran_chance.html",  # Change to real URL or omit for site-wide
    limit=10  # Show top 10 countries
)

## Schema

In [ ]:
#| export

def print_schema_check(result: dict) -> None:
    "Print schema validation result as a rich table."
    summary = result["summary"]
    status_color = "green" if summary["has_google_supported"] else "yellow"
    rprint(f"\n[{status_color}]URL: {result['url']} (HTTP {result['fetch_status']})[/{status_color}]")
    rprint(f"Schemas found: {summary['total_schemas']} | Valid: {summary['valid_count']} | Types: {', '.join(summary['types_found']) or 'none'}\n")

    for s in result["schemas_found"]:
        color = "green" if s["is_valid"] else "red"
        rprint(f"[{color}]{s['type']}[/{color}] ({s['format']}) — Google supported: {s['google_supported']}")
        if s["fields_missing_required"]:
            rprint(f"  [red]Missing required: {', '.join(s['fields_missing_required'])}[/red]")
        if s["fields_missing_recommended"]:
            rprint(f"  [yellow]Missing recommended: {', '.join(s['fields_missing_recommended'])}[/yellow]")
        for w in s["warnings"]:
            rprint(f"  [yellow]Warning: {w}[/yellow]")
@call_parse
def seo_otter_schema_check(
    url: str,  # Live page URL to validate
    timeout: int = 10  # Request timeout in seconds
):
    "Validate schema.org markup on a live page"
    result = validate_page(url, timeout=timeout)
    print_schema_check(result)

In [ ]:
#| export

def print_faq(rows: list[dict], faqs: list[str]) -> None:
    faq_set = set(faqs)
    faq_rows = [r for r in rows if r.get("query") in faq_set]
    table = Table(title="FAQ Queries")
    table.add_column("Impressions", justify="right")
    table.add_column("Clicks", justify="right")
    table.add_column("Question", style="cyan", justify="right")

    for r in faq_rows:
        table.add_row(r["query"], str(r["total_impressions"]), str(r["total_clicks"]))


    rprint(table) if faq_rows else rprint("[yellow]No FAQ queries found[/yellow]")


@call_parse
def seo_otter_faq(
    website_id: int,           # Website ID from the database
    page_url: str = "",        # Optional page URL to filter
    range_type: str = "entire_history",  # Date range type
    limit: int = 50            # Max queries to scan
):
    "Extract FAQ question queries from GSC data for a site or page"
    with get_session() as session:
        website = session.get(Website, website_id)
        if not website:
            rprint(f"[red]❌ Website {website_id} not found[/red]")
            return
        domain = website.url.removeprefix("https://").removeprefix("http://").rstrip("/")
        site_url = f"sc-domain:{domain}"
        start, end = get_date_range(range_type=range_type)
        rows = get_top_queries(session, site_url, start, end,
                               page_path=page_url or None, limit=limit, sort_by="impressions")
        faqs = extract_faq_queries(rows)

    print_faq(rows, faqs)

